In [1]:
# packages
import pandas as pd
from mod02_build_bot_predictor import train_model

### Define a function to extract predictions from the model

In [2]:
def predict_bot(df, model=None):
    """
    Predict whether each account is a bot (1) or human (0).
    """
    if model is None:
        model = train_model()

    preds = model.predict(df)
    return pd.Series(preds, index=df.index)

### Define a function to evaluate model error

In [3]:
def confusion_matrix_and_metrics(y_true, y_pred):
    """
    Computes confusion matrix and common error rates for binary classification.

    Assumes labels:
      0 = negative class
      1 = positive class

    Returns:
      dict with:
        tn, fp, fn, tp
        misclassification_rate
        false_positive_rate
        false_negative_rate
    """
    tn = fp = fn = tp = 0

    for yt, yp in zip(y_true, y_pred):
        if yt == 0 and yp == 0:
            tn += 1
        elif yt == 0 and yp == 1:
            fp += 1
        elif yt == 1 and yp == 0:
            fn += 1
        elif yt == 1 and yp == 1:
            tp += 1
        else:
            raise ValueError("Labels must be 0 or 1")

    total = tn + fp + fn + tp

    misclassification_rate = (fp + fn) / total if total > 0 else 0.0
    false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    false_negative_rate = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    return {
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "misclassification_rate": misclassification_rate,
        "false_positive_rate": false_positive_rate,
        "false_negative_rate": false_negative_rate,
    }


### Load the data

In [4]:
TRAIN_PATH = "mod02_data/train.csv"
train = pd.read_csv(TRAIN_PATH)

TEST_PATH = "mod02_data/test.csv"
test = pd.read_csv(TEST_PATH)

### Format the data by independent vs. dependent variables

In [5]:
X_train = train.drop(columns=["is_bot"])
y_train = train['is_bot']

X_test = test.drop(columns=["is_bot"])
y_test = test['is_bot']

### Build the model on training data

In [6]:
model = train_model(X_train, y_train)

### Get the model predictions on training and test data

In [7]:
y_pred_train = predict_bot(X_train, model)
y_pred_test = predict_bot(X_test, model)

### Check results on the training set (data used to build the model)

In [8]:
confusion_matrix_and_metrics(y_train, y_pred_train)

{'tp': 78,
 'tn': 2606,
 'fp': 31,
 'fn': 285,
 'misclassification_rate': 0.10533333333333333,
 'false_positive_rate': 0.011755783086841108,
 'false_negative_rate': 0.7851239669421488}

### Check results on the test set (new data not yet seen by the model)

In [9]:
confusion_matrix_and_metrics(y_test, y_pred_test)

{'tp': 26,
 'tn': 867,
 'fp': 7,
 'fn': 100,
 'misclassification_rate': 0.107,
 'false_positive_rate': 0.008009153318077803,
 'false_negative_rate': 0.7936507936507936}

# Discussion Questions

### Based on the misclassification rate of your model, discuss your confidence in the ability to predict a bot. 

This model is not a good predictor at finding bot accounts because no matter how I change the classifcations, the flase negative rate is still high (around 70%) and it seems to not know what a bot is. It just says its not a bot and moves on.

### What are potential ramifications of false positives from the model?

If these were real accounts and people, this would cause a lot of actual human users accounts to be flagged as a bot, potentially shutting down a real account and losing trust with that person which can cause them to leave twitter all together. This could cause a company to lose trust will many of their users, not just the ones that are flagged; they dont know if their account will be next. Also if this model automatically shuts down peoples accounts instead of having a real person review the flag, then many users would complain and spread the word on the legitimacy of this company and their review system.

### What are potential ramifications of false negatives from the model?

False negatives could be cause the exact opposite problem, where the platform would then be riddled with too many bots, causing a many user experience when using this platform. This can cause people to interact with fake people and possibly lose busniess or have false data when it comes to who is visiting their sites or bussiness pages. Also, if there are too many bots, that can but a strain on actual servers if there is too much unwanted activity.